# 07 — Zero-shot Qwen2.5-7B-Instruct (4-bit, Unsloth) on GSM8K + SVAMP  *(rejection-sampling study, 7B)*

Same prompt, greedy decoding, 512 tokens and answer extraction as `01_zeroshot` (0.5B), so numbers compare directly.
Model: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (standard bitsandbytes 4-bit), fp16 compute (T4 has no bf16).
Eval runs as **2 processes, one per GPU** (`code/eval7b.py`), each answering every 2nd problem.

Inputs: `00_data`; optional `01`, `05`, `06` outputs for the comparison table. Shared code: repo `code/` (cloned below).

In [1]:
DEBUG = False

## Install
`unsloth` may replace Kaggle's torch / transformers. `ERROR: pip's dependency resolver ...` lines about packages this
notebook doesn't use (cudf, cuml, numba, ...) are warnings. The check runs in a separate process (no torch loaded here).

In [2]:
!pip install -q unsloth
!python -c "import unsloth, torch, transformers, trl; print('unsloth', unsloth.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__); print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.11.4 | torch 2.11.0+cu130 | transformers 5.17.0 | trl 1.15.0
['Tesla T4', 'Tesla T4']


## Setup

In [3]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

# shared code lives in the repo's code/ folder (push your changes before running)
REPO = Path("/tmp/reward-signals-SRM")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/pyther-hub/reward-signals-SRM", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "code"))          # import modules by name (never `import code`: stdlib clash)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()

from common import SPLIT_KEYS, get_logger, save_json
from eval7b import run_sharded_eval

DATA_DIR = Path("/kaggle/input/notebooks/tensorpanda231/00-data-math-reasoning/data")
RUNS_05B = {   # optional 0.5B runs for the comparison (attach as inputs)
    "0.5B zero-shot (01)": "/kaggle/input/notebooks/tensorpanda231/01-zeroshot-qwen2-5-0-5b/exp1_zeroshot_0.5b",
    "0.5B RFT-SFT (05)": "/kaggle/input/notebooks/tensorpanda231/05-sft-with-rft-qwen-05b/exp5_sft_rft_0.5b",
    "0.5B GRPO (06)": "/kaggle/input/notebooks/tensorpanda231/06-grpo-0-5b/exp6_grpo_0.5b",
}
OUT_DIR = Path("/kaggle/working/exp7_zeroshot_7b" + ("_debug" if DEBUG else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
log = get_logger(OUT_DIR)

CFG = dict(splits=["gsm8k_val", "gsm8k_test", "svamp_test"], batch_size=48, limit=100 if DEBUG else None, commit=COMMIT)
save_json(CFG, OUT_DIR / "config.json")
log.info(f"config: {CFG}")

10:30:12 | main  | config: {'splits': ['gsm8k_val', 'gsm8k_test', 'svamp_test'], 'batch_size': 48, 'limit': None, 'commit': '7066cae'}


## Evaluate (both GPUs)

In [4]:
results, metrics = run_sharded_eval(DATA_DIR, CFG["splits"], OUT_DIR, limit=CFG["limit"], batch_size=CFG["batch_size"], log=log)
save_json({"eval": metrics, "gpu_hours_total": round(sum(m["gpu_hours"] for m in metrics.values()), 3)}, OUT_DIR / "metrics.json")
pd.DataFrame(metrics).T

[gpu0] 🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
[gpu1] 🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
[gpu0] Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
[gpu0] To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
[gpu0] 🦥 Unsloth Zoo will now patch everything to make training faster!
[gpu1] Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
[gpu1] To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
[gpu1] 🦥 Unsloth Zoo will now patch everything to make training faster!
[gpu0] ==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. T

,n,accuracy,format_rate,mean_tokens,truncated_rate,wall_min,gpu_hours
val,500.0,0.9360,0.9940,225.1,0.0060,14.83,0.494
gsm8k,1319.0,0.9014,0.9901,234.6,0.0099,41.94,1.398
svamp,1000.0,0.9320,0.9940,164.1,0.0060,26.89,0.896


## Compare with the 0.5B runs

In [5]:
def acc_table(runs):
    """runs: {label: run_dir}; reads metrics.json (01-06 / 07-10 formats) -> accuracy table (val / gsm8k / svamp)."""
    rows = {}
    for label, run_dir in runs.items():
        path = Path(run_dir) / "metrics.json"
        if path.exists():
            m = json.loads(path.read_text())
            m = m.get("eval", m)
            rows[label] = {SPLIT_KEYS.get(k, k): v["accuracy"] for k, v in m.items() if isinstance(v, dict) and "accuracy" in v}
        else:
            log.info(f"{label}: no metrics at {path} (attach it as input to compare)")
    return pd.DataFrame(rows).reindex(["val", "gsm8k", "svamp"])

table = acc_table({**RUNS_05B, "7B zero-shot (07)": OUT_DIR})
table.round(4)

,0.5B zero-shot (01),0.5B RFT-SFT (05),0.5B GRPO (06),7B zero-shot (07)
val,NaN,0.6660,0.6980,0.9360
gsm8k,0.4375,0.4936,0.5436,0.9014
svamp,0.5370,0.6190,0.6320,0.9320


## A few wrong answers

In [6]:
for split, df in results.items():
    for _, r in df[~df.correct].head(2).iterrows():
        print(f"=== {split} | gold={r.gold} | pred={r.pred} | format={r.has_format} | tokens={r.n_tokens}\n{r.question}\n--- response ---\n{r.response}\n")

=== gsm8k_val | gold=104 | pred=40 | format=False | tokens=512
In a field where there are 200 animals, there are 40 cows, 56 sheep and goats. How many goats are there?
--- response ---
To solve this problem, we need to determine the number of goats in the field. Let's break it down step by step.

1. **Total number of animals**: We know there are 200 animals in total.
2. **Number of cows**: There are 40 cows.
3. **Number of sheep and goats combined**: The problem states that there are 56 sheep and goats combined.

Let's denote:
- \( C \) as the number of cows,
- \( S \) as the number of sheep,
- \( G \) as the number of goats.

From the information given:
\[ C = 40 \]
\[ S + G = 56 \]

We also know the total number of animals is the sum of cows, sheep, and goats:
\[ C + S + G = 200 \]

Substituting the known values into the equation:
\[ 40 + S + G = 200 \]

Since \( S + G = 56 \), we can substitute \( 56 \) for \( S + G \):
\[ 40 + 56 = 200 - (S + G) \]

This simplifies to:
\[ 96 = 200 